# Explainability Track — Step 1
## XGBoost / Random Forest Classifier: Training, Tuning, Gene Importance

**Owner:** Namitha
**Input files (upload these to the Colab session):**
- `X_train_selected.csv`, `X_test_selected.csv` — 300 selected genes, z-scored
- `y_train.csv`, `y_test.csv` — binary labels
- `selected_genes.pkl` — list of the 300 selected gene probe IDs
- `mutual_information_scores.csv` — full MI ranking (for reference/reporting only)

**What this notebook produces (used later by SHAP + `/explain` backend):**
- `xgb_model.pkl`, `rf_model.pkl` — trained, tuned models
- `xgb_gene_importance.csv`, `rf_gene_importance.csv` — ranked gene importance tables
- Cross-validated performance numbers for both models (rough sanity check — full formal comparison happens in your Evaluation step)


## 1. Setup & Imports

In [ ]:
!pip install xgboost shap -q

import pandas as pd
import numpy as np
import pickle
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import xgboost as xgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Upload & Load Data
Run the cell below, then use the file picker to upload all 6 files
(`X_train_selected.csv`, `X_test_selected.csv`, `y_train.csv`, `y_test.csv`,
`selected_genes.pkl`, `mutual_information_scores.csv`).


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
X_train = pd.read_csv("X_train_selected.csv")
X_test  = pd.read_csv("X_test_selected.csv")
y_train = pd.read_csv("y_train.csv").iloc[:, 0]
y_test  = pd.read_csv("y_test.csv").iloc[:, 0]

with open("selected_genes.pkl", "rb") as f:
    selected_genes = pickle.load(f)

mi_scores = pd.read_csv("mutual_information_scores.csv", index_col=0)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train distribution:\n", y_train.value_counts())
print("y_test distribution:\n",  y_test.value_counts())


## 3. Sanity Checks
Always verify alignment before training — a silent row-mismatch between `X` and `y`
will train a model that looks fine but is meaningless.


In [ ]:
assert X_train.shape[0] == y_train.shape[0], "Row mismatch: X_train vs y_train"
assert X_test.shape[0]  == y_test.shape[0],  "Row mismatch: X_test vs y_test"
assert list(X_train.columns) == list(X_test.columns), "Train/test columns differ"
assert list(X_train.columns) == list(selected_genes), "Columns don't match selected_genes.pkl"
assert X_train.isna().sum().sum() == 0, "NaNs found in X_train"
assert X_test.isna().sum().sum() == 0, "NaNs found in X_test"

print("All checks passed.")
print(f"Train class balance: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Test class balance:  {y_test.value_counts(normalize=True).to_dict()}")
print("Note: train is SMOTE-balanced, test is left imbalanced on purpose — this is correct.")


## 4. Baseline Models (no tuning)
Train quick baselines first — this validates the pipeline end-to-end before
spending time on hyperparameter search, and gives you a reference point to
confirm tuning actually helped.


In [ ]:
# --- Baseline XGBoost ---
xgb_base = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_base.fit(X_train, y_train)
xgb_base_preds = xgb_base.predict(X_test)
xgb_base_proba = xgb_base.predict_proba(X_test)[:, 1]

print("=== Baseline XGBoost ===")
print("Accuracy:", accuracy_score(y_test, xgb_base_preds))
print("F1:      ", f1_score(y_test, xgb_base_preds))
print("ROC-AUC: ", roc_auc_score(y_test, xgb_base_proba))


In [ ]:
# --- Baseline Random Forest ---
rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_base.fit(X_train, y_train)
rf_base_preds = rf_base.predict(X_test)
rf_base_proba = rf_base.predict_proba(X_test)[:, 1]

print("=== Baseline Random Forest ===")
print("Accuracy:", accuracy_score(y_test, rf_base_preds))
print("F1:      ", f1_score(y_test, rf_base_preds))
print("ROC-AUC: ", roc_auc_score(y_test, rf_base_proba))


## 5. Hyperparameter Tuning

Using `RandomizedSearchCV` with **stratified k-fold** (important given the class
imbalance in the test set — stratification keeps fold proportions consistent).
Scoring on ROC-AUC since it's threshold-independent and works well for
imbalanced/biomarker-style classification.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

xgb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.2, 0.3],
}

xgb_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                 random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=xgb_param_dist,
    n_iter=50,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
xgb_search.fit(X_train, y_train)

print("Best XGBoost params:", xgb_search.best_params_)
print("Best CV ROC-AUC:", xgb_search.best_score_)

xgb_tuned = xgb_search.best_estimator_


In [ ]:
rf_param_dist = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "bootstrap": [True, False],
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=50,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_train, y_train)

print("Best RF params:", rf_search.best_params_)
print("Best CV ROC-AUC:", rf_search.best_score_)

rf_tuned = rf_search.best_estimator_


## 6. Cross-Validated Sanity Check on Tuned Models
This is a lightweight stability check, not the full formal evaluation
(that's a separate step where you'll compare both models properly with
confusion matrices etc. across the whole team's metrics).


In [ ]:
for name, model in [("XGBoost (tuned)", xgb_tuned), ("Random Forest (tuned)", rf_tuned)]:
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name}: CV ROC-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


In [ ]:
# Held-out test set performance (quick check, not final report)
print("=== Tuned XGBoost on test set ===")
preds = xgb_tuned.predict(X_test)
proba = xgb_tuned.predict_proba(X_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, preds))
print("F1:      ", f1_score(y_test, preds))
print("ROC-AUC: ", roc_auc_score(y_test, proba))
print(classification_report(y_test, preds))

print("\n=== Tuned Random Forest on test set ===")
preds = rf_tuned.predict(X_test)
proba = rf_tuned.predict_proba(X_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, preds))
print("F1:      ", f1_score(y_test, preds))
print("ROC-AUC: ", roc_auc_score(y_test, proba))
print(classification_report(y_test, preds))


## 7. Gene Importance (built-in model importances)

This is the **first-pass** gene ranking, using each model's native importance
score. It's a good sanity check before SHAP (your next step) — the two rankings
should broadly agree. Native importances are faster but less rigorous than SHAP,
which is why SHAP is the "real" biomarker ranking in your pipeline.


In [ ]:
xgb_importance = pd.DataFrame({
    "gene": X_train.columns,
    "importance": xgb_tuned.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

rf_importance = pd.DataFrame({
    "gene": X_train.columns,
    "importance": rf_tuned.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 15 genes — XGBoost:")
display(xgb_importance.head(15))

print("\nTop 15 genes — Random Forest:")
display(rf_importance.head(15))


In [ ]:
# Optional: cross-reference against the original Mutual Information ranking
# to see if top genes from the model agree with the univariate MI signal
top_xgb_genes = xgb_importance.head(20)["gene"].tolist()
mi_check = mi_scores.loc[mi_scores.index.intersection(top_xgb_genes)].sort_values("mi_score", ascending=False)
print("MI scores for XGBoost's top 20 genes:")
display(mi_check)


## 8. Save Everything

These files feed directly into:
- Your **SHAP analysis** step (load the models, run `TreeExplainer` on them)
- Your **Evaluation** step (formal accuracy/F1/ROC-AUC/confusion matrix comparison)
- Your **`/explain` backend endpoint** (loads `xgb_model.pkl` at minimum)

Download them from the Colab file browser (left sidebar) or mount Google Drive
to save automatically.


In [ ]:
joblib.dump(xgb_tuned, "xgb_model.pkl")
joblib.dump(rf_tuned, "rf_model.pkl")

xgb_importance.to_csv("xgb_gene_importance.csv", index=False)
rf_importance.to_csv("rf_gene_importance.csv", index=False)

print("Saved: xgb_model.pkl, rf_model.pkl, xgb_gene_importance.csv, rf_gene_importance.csv")

# Uncomment to download directly in Colab:
# from google.colab import files
# for f in ["xgb_model.pkl", "rf_model.pkl", "xgb_gene_importance.csv", "rf_gene_importance.csv"]:
#     files.download(f)


## Next Steps
1. **SHAP analysis** — load `xgb_model.pkl` (and/or `rf_model.pkl`) with `shap.TreeExplainer`, generate summary plots, and produce the final biomarker gene ranking (this supersedes the native-importance ranking above).
2. **Evaluation** — formal accuracy/F1/ROC-AUC/confusion matrix comparison across both models, written up as a comparison report.
3. **Backend `/explain` endpoint** — load the saved model + SHAP explainer, serve top genes + plot data as JSON.
